In [ ]:
!nvidia-smi

Tue Apr 28 09:45:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/VuTrinhNguyenHoang/bidirectional-table-text-alignment.git

Cloning into 'bidirectional-table-text-alignment'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 175 (delta 79), reused 138 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 45.19 KiB | 3.23 MiB/s, done.
Resolving deltas: 100% (79/79), done.


In [3]:
%cd bidirectional-table-text-alignment

/kaggle/working/bidirectional-table-text-alignment


In [4]:
!mkdir -p outputs/checkpoints outputs/data/processed
!cp -a /kaggle/input/models/trnhvu2/bitt/pytorch/default/1/checkpoints/. outputs/checkpoints/
!cp -a /kaggle/input/models/trnhvu2/bitt/pytorch/default/1/data/processed/. outputs/data/processed/

In [5]:
!pip -q install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00


In [6]:
from pathlib import Path
import json

MODE = "small"
GENERATOR_MODE = "full"
SELECTOR_MODE = "small"
CONFIG = "configs/main.yaml"
TOP_K = 3
THRESHOLD_STEP = 0.01

In [7]:
!PYTHONPATH=. python3 scripts/evaluation_generation.py --mode {MODE} --generator-mode {GENERATOR_MODE} --config {CONFIG}

Loading weights: 100%|█| 131/131 [00:00<00:00, 1672.18it/s, Materializing param=
Generating: 100%|███████████████████████████| 1000/1000 [04:55<00:00,  3.38it/s]
{
  "bleu": {
    "score": 35.704330433471824,
    "counts": [
      11218,
      6690,
      4455,
      3096
    ],
    "totals": [
      16530,
      15530,
      14530,
      13530
    ],
    "precisions": [
      67.86448880822746,
      43.07791371538957,
      30.660701995870614,
      22.88248337028825
    ],
    "bp": 0.9434638618046693,
    "sys_len": 16530,
    "ref_len": 17492
  },
  "rouge": {
    "rouge1": 0.6843396361975618,
    "rouge2": 0.45889314158218864,
    "rougeL": 0.5876008928932069,
    "rougeLsum": 0.5869789750168872
  },
  "mode": "small",
  "generator_mode": "full"
}


In [8]:
!PYTHONPATH=. python3 scripts/tune_cell_selector_threshold.py --mode {MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k {TOP_K} --split valid --threshold_step {THRESHOLD_STEP}

Loading weights: 100%|█| 104/104 [00:00<00:00, 1626.42it/s, Materializing param=
Sweeping thresholds: 100%|████████████████████| 101/101 [00:07<00:00, 12.85it/s]
{
  "mode": "small",
  "split": "valid",
  "selector_mode": "small",
  "top_k": 3,
  "selection_strategy": "threshold_with_top_k_fallback",
  "best_threshold": 0.88,
  "best_metrics": {
    "cell_precision": 0.7886666666666666,
    "cell_recall": 0.7059697170019996,
    "cell_f1": 0.7155285832230953,
    "cell_exact_match": 0.294,
    "threshold": 0.88,
    "top_k": 3,
    "fallback_rate": 0.523,
    "empty_threshold_rate": 0.001,
    "overflow_threshold_rate": 0.522,
    "avg_selected_count": 2.633
  },
  "results": [
    {
      "cell_precision": 0.691,
      "cell_recall": 0.7232430503353329,
      "cell_f1": 0.6644524537969659,
      "cell_exact_match": 0.144,
      "threshold": 0.0,
      "top_k": 3,
      "fallback_rate": 1.0,
      "empty_threshold_rate": 0.0,
      "overflow_threshold_rate": 1.0,
      "avg_selected_co

In [9]:
tuning_path = Path(f"outputs/metrics/{MODE}/cell_selector_threshold_tuning_valid_top{TOP_K}.json")
tuning = json.loads(tuning_path.read_text(encoding="utf-8"))
BEST_THRESHOLD = tuning["best_threshold"]
BEST_THRESHOLD, tuning["best_metrics"]

(0.88,
 {'cell_precision': 0.7886666666666666,
  'cell_recall': 0.7059697170019996,
  'cell_f1': 0.7155285832230953,
  'cell_exact_match': 0.294,
  'threshold': 0.88,
  'top_k': 3,
  'fallback_rate': 0.523,
  'empty_threshold_rate': 0.001,
  'overflow_threshold_rate': 0.522,
  'avg_selected_count': 2.633})

In [11]:
!PYTHONPATH=. python3 scripts/evaluation_e2e.py --mode {MODE} --generator-mode {GENERATOR_MODE} --selector-mode {SELECTOR_MODE} --config {CONFIG} --top_k {TOP_K} --threshold {BEST_THRESHOLD}

Loading weights: 100%|█| 131/131 [00:00<00:00, 1714.37it/s, Materializing param=
Loading weights: 100%|█| 104/104 [00:00<00:00, 1641.83it/s, Materializing param=
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 541kB/s]
README.md: 10.5kB [00:00, 29.4MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 309kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 2.83MB/s]
model.safetensors: 100%|████████████████████| 90.9M/90.9M [00:00<00:00, 111MB/s]
Loading weights: 100%|█| 103/103 [00:00<00:00, 1737.95it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 2.58MB/s]

In [12]:
threshold_tag = f"threshold{BEST_THRESHOLD:.4f}".replace(".", "p")
metric_path = Path(f"outputs/metrics/{MODE}/e2e_metrics_{threshold_tag}_top{TOP_K}.json")
metrics = json.loads(metric_path.read_text(encoding="utf-8"))
metrics

{'mode': 'small',
 'generator_mode': 'full',
 'selector_mode': 'small',
 'top_k': 3,
 'threshold': 0.88,
 'selection_strategy': 'threshold_with_top_k_fallback',
 'cell_selection': {'cell_precision': 0.7886666666666666,
  'cell_recall': 0.7059697170019996,
  'cell_f1': 0.7155285832230953,
  'cell_exact_match': 0.294},
 'consistency': {'lexical': {'bleu': {'score': 28.422099113421453,
    'counts': [9630, 5449, 3484, 2307],
    'totals': [14258, 13258, 12258, 11258],
    'precisions': [67.541029597419,
     41.09971338060039,
     28.422254853972916,
     20.49209451057026],
    'bp': 0.7970642163445462,
    'sys_len': 14258,
    'ref_len': 17492},
   'rouge': {'rouge1': 0.6304237955624694,
    'rouge2': 0.4094433632522637,
    'rougeL': 0.5422558761113856,
    'rougeLsum': 0.5425263791125937}},
  'semantic': {'cosine_mean': 0.8151534632444382,
   'cosine_std': 0.14038121872123635,
   'cosine_min': 0.250560462474823,
   'cosine_max': 1.0000001192092896},
  'number_faithfulness': {'n_exam